# ⚔️ CRUSADER — F03 SIGISMUND
## Rendu Remotion → short_render.mp4

> *"Sigismund stood unmoved, and from him radiated the Emperor's will."*

---

**Ce notebook tourne sur CPU — GPU non requis.**

### Étapes :
1. Montage Google Drive
2. Installation Node.js + Chrome
3. Téléchargement des scripts et sources Remotion
4. Configuration des chemins
5. Validation CUSTOS check-out
6. Setup Remotion + npm install
7. Rendu vidéo
8. Validation CUSTOS check-in
9. Aperçu et téléchargement

---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 2 — Installation Node.js 20 + Chrome

In [ ]:
import subprocess, sys

# ── Node.js 20 ────────────────────────────────────────────────────────────────
print('Installation Node.js 20...')
subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash -', shell=True, check=True)
subprocess.run(['apt-get', 'install', '-y', 'nodejs'], check=True)

result = subprocess.run(['node', '--version'], capture_output=True, text=True)
print(f'[OK] Node.js : {result.stdout.strip()}')

result = subprocess.run(['npm', '--version'], capture_output=True, text=True)
print(f'[OK] npm     : {result.stdout.strip()}')

# ── Chromium (pour Remotion headless) ────────────────────────────────────────
print('\nInstallation Chromium...')
subprocess.run(['apt-get', 'install', '-y', 'chromium-browser'], check=True)

result = subprocess.run(['which', 'chromium-browser'], capture_output=True, text=True)
chrome_path = result.stdout.strip()
print(f'[OK] Chrome  : {chrome_path}')

---
## Étape 3 — Téléchargement des sources Remotion depuis GitHub

In [ ]:
import urllib.request, os, json

REPO_RAW   = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
PROJECT_DIR = '/content/crusader'

files_to_download = [
    ('F03_SIGISMUND/CODEBASE/package.json',              'package.json'),
    ('F03_SIGISMUND/CODEBASE/remotion.config.js',        'remotion.config.js'),
    ('F03_SIGISMUND/CODEBASE/src/index.jsx',             'src/index.jsx'),
    ('F03_SIGISMUND/CODEBASE/src/Root.jsx',              'src/Root.jsx'),
    ('F03_SIGISMUND/CODEBASE/src/Main.jsx',              'src/Main.jsx'),
    ('F03_SIGISMUND/CODEBASE/src/components/Scene.jsx',  'src/components/Scene.jsx'),
    ('F03_SIGISMUND/CODEBASE/src/components/Subtitle.jsx', 'src/components/Subtitle.jsx'),
    ('F03_SIGISMUND/CODEBASE/src/components/Background.jsx', 'src/components/Background.jsx'),
    ('F03_SIGISMUND/CODEBASE/crs_f03_sigismund.py',      'crs_f03_sigismund.py'),
    ('CRS_CUSTOS.py',                                    'CRS_CUSTOS.py'),
]

for rel_path, dest_name in files_to_download:
    dest_path = os.path.join(PROJECT_DIR, dest_name)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    urllib.request.urlretrieve(f'{REPO_RAW}/{rel_path}', dest_path)
    print(f'[OK] {dest_name}')

os.makedirs(os.path.join(PROJECT_DIR, 'public', 'images'), exist_ok=True)
print('\nTous les fichiers téléchargés.')

---
## Étape 4 — Configuration des chemins

> **Modifiez `DRIVE_BASE` si votre structure Google Drive est différente.**

In [ ]:
import os

# ── MODIFIEZ ICI SI NÉCESSAIRE ─────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_CRUSADER'
# ──────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = '/content/crusader'
F03_IN      = os.path.join(DRIVE_BASE, 'F03_SIGISMUND', 'IN')
F03_OUT     = os.path.join(DRIVE_BASE, 'F03_SIGISMUND', 'OUT')
os.makedirs(F03_OUT, exist_ok=True)

print('Configuration :')
print(f'  F03 IN  : {F03_IN}')
print(f'  F03 OUT : {F03_OUT}')
print(f'  Projet  : {PROJECT_DIR}')
print()

checks = {
    'timing.json':   os.path.isfile(os.path.join(F03_IN, 'timing.json')),
    'roadmap.json':  os.path.isfile(os.path.join(F03_IN, 'roadmap.json')),
    'audio_clean.mp3': os.path.isfile(os.path.join(F03_IN, 'audio_clean.mp3')),
    'images/':       os.path.isdir(os.path.join(F03_IN, 'images')),
}
for name, ok in checks.items():
    print(f'  {name}: {"✓" if ok else "✗ MANQUANT"}')

---
## Étape 5 — Validation CUSTOS check-out (F03)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F03', '--mode', 'check-out', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-out FAIL. Vérifiez les fichiers dans F03/IN/.')

---
## Étape 6 — npm install

> **Première exécution : ~2-3 min. Exécutions suivantes : rapide (cache npm).**

In [ ]:
import subprocess, os

node_modules = os.path.join(PROJECT_DIR, 'node_modules')
if not os.path.isdir(node_modules):
    print('npm install en cours...')
    subprocess.run(['npm', 'install'], cwd=PROJECT_DIR, check=True)
    print('[OK] Packages installés.')
else:
    print('[OK] node_modules existant — installation ignorée.')

---
## Étape 7 — Rendu vidéo

> **Durée estimée : 1-5 min selon la longueur de la vidéo.**
>
> Le rendu utilise `--gl swangle` (rendu logiciel) pour compatibilité Colab.

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'crs_f03_sigismund.py'),
     '--input',   F03_IN,
     '--output',  F03_OUT,
     '--project', PROJECT_DIR,
     '--gl',      'swangle'],
)
if result.returncode != 0:
    print('[STOP] Rendu échoué. Consultez les logs ci-dessus.')

---
## Étape 8 — Validation CUSTOS check-in (F03)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F03', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-in FAIL. short_render.mp4 absent ou trop petit.')
else:
    print('[OK] short_render.mp4 validé — prêt pour transfert vers F04.')

---
## Étape 9 — Aperçu et téléchargement

In [ ]:
import os
from IPython.display import Video, display

output_path = os.path.join(F03_OUT, 'short_render.mp4')

if os.path.isfile(output_path):
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f'short_render.mp4 — {size_mb:.1f} MB')
    print(f'Chemin : {output_path}')
    print()
    display(Video(output_path, embed=True, width=360))
else:
    print('[ERREUR] short_render.mp4 introuvable.')

In [ ]:
# Téléchargement direct depuis Colab (optionnel)
from google.colab import files
files.download(output_path)